In [16]:
import sqlite3
from langgraph.graph import StateGraph, START, END 
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver 
from langgraph.types import interrupt, Command 


llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect(
    "memory.db",
    check_same_thread = False,
)

config = {
    "configurable": {
        "thread_id": "2",
    },
}

In [17]:
class State(MessagesState):
    pass 

graph_builder = StateGraph(State)


In [18]:
def chatbot(state:State):
    response = llm.invoke(state["messages"])
    return {
        "messages": [response]
    }

    

In [19]:
graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot",END)

graph=graph_builder.compile(
    checkpointer = SqliteSaver(conn),
)




In [20]:
result = graph.invoke(
    {
        "messages":[
            {"role":"user","content":"Hello my name is Hyeseon"},
        ]
    },
    config = config,
)

KeyError: 'thread_id'

In [ ]:
for message in result["messages"]:
    message.pretty_print()
    